# 저작권 판례 RAG 단계별 개선 실험

## 실험 목적

어제 구현한 Baseline RAG를 보존하고, Day 2에서 배운 기법을 문제 원인에 맞춰 하나씩 적용한다.

1. Baseline Vector Search 결과를 측정한다.
2. 판례 청크의 문맥 부족을 `Context-Enriched Chunking`으로 개선한다.
3. 동일 판례의 중복 검색을 `MMR + 사건번호 중복 제거`로 개선한다.
4. 법률 용어와 사건번호 검색을 `BM25 + Vector Hybrid Search`로 보완한다.
5. 동일한 평가 질문으로 각 단계의 효과와 한계를 비교한다.

> 중요한 것은 기법의 개수가 아니라 `문제 관찰 → 원인 진단 → 기법 적용 → 동일 조건 비교`의 과정이다.


## 전체 실험 워크플로우

```mermaid
flowchart LR
    Q[사용자 질문] --> BASE[Baseline Vector Search]
    Q --> BM[BM25 Search]
    Q --> EV[Context-Enriched Vector Search]
    EV --> MMR[MMR 후보 검색]
    BM --> RRF[RRF 순위 결합]
    MMR --> RRF
    RRF --> DEDUP[사건번호 중복 제거]
    DEDUP --> CTX[Top-3 Context 구성]
    CTX --> LLM[LLM 답변 생성]
    LLM --> OUT[답변과 근거 판례 출력]

    BASE -. 단계별 비교 .-> METRIC[Hit@3 · MRR · 중복률]
    EV -. 단계별 비교 .-> METRIC
    MMR -. 단계별 비교 .-> METRIC
    DEDUP -. 단계별 비교 .-> METRIC
```

실험은 Baseline을 삭제하거나 대체하지 않는다. 같은 질문을 사용해 `Baseline → Enriched → MMR → Hybrid` 순서로 효과를 측정한다.


In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma chromadb openpyxl pandas rank-bm25


## Preparation 1. API Key 설정

In [ ]:
import os
from google.colab import userdata

# Colab 왼쪽의 열쇠 아이콘(Secrets)에 OPENAI_API_KEY 등록 필요
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Colab Secrets에 OPENAI_API_KEY를 등록해 주세요.")
os.environ["OPENAI_API_KEY"] = api_key

print("API Key 설정 완료")

## Preparation 2. 판례 데이터 업로드

`Copyright_case_law_Supreme.xlsx` 파일을 업로드


In [ ]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded))

print(f"Uploaded file: {DATA_PATH}")


## Preparation 3. 데이터 확인

In [ ]:
import pandas as pd

df = pd.read_excel(DATA_PATH)

print(f"판례 수: {len(df):,}")
print(f"컬럼: {df.columns.tolist()}")

display(df.head())


# Indexing : Knowledge Base 구축

## Step1. Document Loader

Excel의 각 판례 행을 Document 객체로 변환

- `page_content` : 판례 본문
- `metadata` : 사건명, 사건번호, 선고일자, 법원명

Metadata를 유지하면 이후 검색 결과에서 어떤 판례가 검색되었는지 추적할 수 있음


In [ ]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    content = str(row["판례내용"])

    metadata = {
        "case_name": str(row["사건명"]),
        "case_no": str(row["사건번호"]),
        "decision_date": str(row["선고일자"]),
        "court": str(row["법원명"]),
    }

    documents.append(
        Document(
            page_content=content,
            metadata=metadata
        )
    )

print(f"Documents: {len(documents):,}")
print("\n[Metadata Sample]")
print(documents[0].metadata)
print("\n[Content Sample]")
print(documents[0].page_content[:500])


## Step2. Text Splitter — Baseline Chunking

판례 본문 Chunk Baseline 설정으로 시작

- `chunk_size = 1000`
- `chunk_overlap = 150`
  
> Chunk Size / Overlap / 구조 기반 Chunking 등을 개선 대상으로 실험할 수 있음


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
)

docs = text_splitter.split_documents(documents)

print(f"Original Documents: {len(documents):,}")
print(f"Chunks: {len(docs):,}")

print("\n[Chunk Sample]")
print(docs[0].page_content[:500])

print("\n[Metadata after Chunking]")
print(docs[0].metadata)


## Step3. Embedding

판례 Chunk를 의미를 표현하는 Vector로 변환

- Model: `text-embedding-3-small`


In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding Model 설정 완료")


## Step4. Vector Store — ChromaDB

별도의 `persist_directory`를 사용하지 않고 현재 Colab 런타임에서만 사용하는 ChromaDB를 생성

셀을 다시 실행할 때 기존 Collection과 충돌하거나 문서가 중복 적재되지 않도록  
실행할 때마다 새로운 Collection 이름을 생성

> Colab 런타임이 종료시 이 Vector DB도 함께 사라짐


In [ ]:
import uuid
from langchain_chroma import Chroma

# 셀을 다시 실행할 때마다 새로운 Collection 생성(중복적재 방지)
collection_name = f"copyright_baseline_{uuid.uuid4().hex[:8]}"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name=collection_name,
)

print("ChromaDB 생성 완료")
print(f"Collection: {collection_name}")
print(f"저장된 벡터 수: {vectorstore._collection.count():,}")


# Runtime

## Step5. Retriever

Baseline에서는 Vector Similarity Search + Top-k 3 사용


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

print("Retriever 설정 완료: similarity / top-k = 3")


### Retrieval 결과 확인하기

In [ ]:
query = "온라인 게임의 규칙이나 전개 방식이 비슷하면 저작권 침해인가요?"

retrieved_docs = retriever.invoke(query)

print(f"질문: {query}")
print(f"\n검색된 Chunk 수: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs, 1):
    print("=" * 80)
    print(
        f"[{i}] {doc.metadata.get('case_name')} "
        f"/ {doc.metadata.get('case_no')} "
        f"/ {doc.metadata.get('decision_date')}"
    )
    print(doc.page_content[:800])


## Step6. Prompt Construction

검색된 판례 Chunk를 Context로 넣고, 질문과 답변 지침을 함께 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 저작권 판례를 분석하는 AI Assistant입니다.

[Instruction]
아래 [참고 판례]만을 근거로 질문에 답변하세요.
근거가 충분하지 않으면 추측하지 말고
"제공된 판례만으로는 충분한 근거를 찾기 어렵습니다."라고 답하세요.

답변에는 핵심 판단과 그 근거를 간결하게 설명하세요.

[Context]
{context}

[Question]
{question}

[Answer]
""")

def format_docs(docs):
    return "\n\n".join(
        f"[사건명: {d.metadata.get('case_name')} | "
        f"사건번호: {d.metadata.get('case_no')} | "
        f"선고일자: {d.metadata.get('decision_date')}]\n"
        f"{d.page_content}"
        for d in docs
    )

print("Prompt Template 구성 완료")


### What Does the LLM Actually See?

Retriever가 가져온 판례 Chunk가 실제 Prompt의 Context에 어떻게 들어가는지 확인


In [ ]:
final_prompt = prompt.invoke({
    "context": format_docs(retrieved_docs),
    "question": query,
})

print(final_prompt.messages[0].content)


## Step7. LLM

검색된 Context를 바탕으로 답변을 생성할 LLM을 설정


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.2",
    temperature=0.2,
)

print("LLM 설정 완료")


## Step8. Chain & Pipeline

Retriever → Prompt → LLM → Output Parser를 하나의 실행 흐름으로 연결


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Baseline RAG Chain 구성 완료")


## Baseline RAG 테스트

In [ ]:
question = "제품 사진도 모두 저작권으로 보호받을 수 있나요?"

answer = rag_chain.invoke(question)

print("[Question]")
print(question)
print("\n[Answer]")
print(answer)


## 응답과 Retrieval 근거 함께 확인하기

답변 품질을 볼 때는 최종 답변만 보지 말고 Retrieval 결과가 적절했는지 먼저 확인


In [ ]:
def ask_rag(question):
    retrieved = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    print("[Question]")
    print(question)

    print("\n[Answer]")
    print(answer)

    print("\n[Retrieved Cases]")
    for i, doc in enumerate(retrieved, 1):
        print(
            f"{i}. {doc.metadata.get('case_name')} "
            f"({doc.metadata.get('case_no')}, {doc.metadata.get('decision_date')})"
        )

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved,
    }

result = ask_rag(
    "저작자가 동의한 범위 안에서 저작물을 수정해도 문제가 되나요?"
)


# [Task Guide: Copyright Case Law RAG]

## Baseline Observation — Limits of Basic RAG

먼저 제공된 Baseline RAG의 한계를 확인합니다.

- 관련 판례가 Top-k 안에 검색되는가?
- 관련 판례가 검색되었지만 필요한 판단 부분이 Chunk 경계에서 잘리지는 않는가?
- 서로 다른 사건의 Chunk가 섞여 답변이 모호해지지는 않는가?
- 질문 표현과 판례 표현이 다른 경우에도 잘 검색되는가?
- 검색은 잘 되었지만 최종 답변 품질이 충분하지 않은 경우가 있는가?

## Challenge — Improve the Baseline

Day2 / Day3에서 학습한 Advanced RAG Technique 중
2개 이상을 적용하여 Baseline의 검색 및 응답 품질을 개선합니다.

예시:
- Chunking Strategy
- Query Transformation / Multi Query
- Hybrid Search
- Metadata Filtering
- Reranking
- Verification / Retry
- ...

단순히 기법을 적용하는 것보다
Baseline의 어떤 문제를 해결하기 위해 해당 기법을 선택했는지 설명하는 것이 중요합니다.

## Evaluation — Did It Really Improve?

동일한 Evaluation Question Set을 사용하여
Baseline과 개선 모델의 결과를 비교합니다.

다음 내용을 중심으로 확인하세요.

- Retrieval 결과는 개선되었는가?
- 관련 판례가 더 정확하게 검색되는가?
- 최종 답변의 정확성 및 근거성이 개선되었는가?
- 적용한 Technique이 모든 질문에 동일하게 효과적이었는가?
- 개선되지 않았거나 오히려 악화된 경우는 무엇인가?

가능한 경우 RAGAS 등의 평가 방법을 활용하여
정량적인 결과도 함께 비교해 보세요.

## Final Workflow

> Baseline  
> → Failure Analysis  
> → Advanced RAG Technique 적용  
> → Before / After Comparison  
> → Evaluation  
> → Lessons Learned


# Advanced RAG 단계별 실험

> 이 절의 5개 문항은 개발 과정에서 만든 정답 판례 기반 sanity check이다. 최종 과제 평가는 아래의 공식 Evaluation Set으로 별도 수행한다.

## 1. 실험 설계

### Baseline에서 확인할 문제

- 같은 사건의 여러 청크가 Top-k를 중복 점유하는가?
- 판례 본문만 임베딩하여 사건명·사건번호·선고일자가 검색 의미에 반영되지 않는가?
- 자연어 질문과 판례의 정확한 법률 용어가 다를 때 핵심 판례를 놓치는가?

### 통제 조건

- 모든 방법에서 같은 원본 데이터와 평가 질문을 사용한다.
- 최종 반환 문서는 Top-3로 통일한다.
- 개선 기법은 이전 단계 위에 하나씩 추가한다.
- 최종 답변보다 Retrieval 결과를 먼저 평가한다.


In [ ]:
# 모든 검색 방법에서 공통으로 사용할 평가 질문과 기대 판례
evaluation_set = [
    {
        "question": "온라인 게임의 규칙이나 전개 방식이 비슷하면 저작권 침해인가요?",
        "expected_case_no": "2017다212095",
        "reason": "게임 규칙과 창작적 표현의 구별",
    },
    {
        "question": "제품 사진도 모두 저작권으로 보호받을 수 있나요?",
        "expected_case_no": "98다43366",
        "reason": "사진저작물의 창작성",
    },
    {
        "question": "저작자가 동의한 범위 안에서 저작물을 수정해도 문제가 되나요?",
        "expected_case_no": "2010다79923",
        "reason": "동일성유지권과 동의 범위",
    },
    {
        "question": "프로그램의 기능이나 아이디어도 저작권으로 보호되나요?",
        "expected_case_no": "2012도12828",
        "reason": "프로그램의 아이디어와 표현 구별",
    },
    {
        "question": "다른 사람의 저작물 링크를 제공하면 저작권 침해를 방조한 것인가요?",
        "expected_case_no": "2017도19025",
        "reason": "링크 제공과 공중송신권 침해 방조",
    },
]

evaluation_df = pd.DataFrame(evaluation_set)
display(evaluation_df)


## 2. 공통 평가 함수

- `Hit@3`: 기대 판례가 Top-3 안에 포함된 질문의 비율
- `MRR`: 기대 판례 순위의 역수 평균. 기대 판례를 찾지 못하면 0점
- `Duplicate Rate`: 전체 검색 슬롯 중 동일 사건번호가 반복된 비율
- `Average Unique Cases`: 질문당 검색된 고유 판례 수

답변 문장이 자연스러워도 검색 근거가 틀릴 수 있으므로 검색 단계부터 측정한다.


In [ ]:
def normalize_case_no(value):
    return str(value or "").strip().replace(" ", "")


def case_no_of(doc):
    return normalize_case_no(doc.metadata.get("case_no", ""))


def evaluate_retrieval(name, retrieve_fn, evaluation_set, top_k=3):
    rows = []
    details = {}

    for item in evaluation_set:
        question = item["question"]
        expected = normalize_case_no(item["expected_case_no"])
        retrieved = list(retrieve_fn(question))[:top_k]
        case_numbers = [case_no_of(doc) for doc in retrieved]
        expected_rank = next(
            (rank for rank, case_no in enumerate(case_numbers, 1) if case_no == expected),
            None,
        )

        rows.append({
            "method": name,
            "question": question,
            "expected_case_no": expected,
            "retrieved_case_nos": " | ".join(case_numbers),
            "hit_at_3": int(expected_rank is not None and expected_rank <= 3),
            "reciprocal_rank": 1 / expected_rank if expected_rank else 0.0,
            "expected_rank": expected_rank,
            "retrieved_count": len(case_numbers),
            "unique_case_count": len(set(case_numbers)),
            "duplicate_count": len(case_numbers) - len(set(case_numbers)),
        })
        details[question] = retrieved

    result_df = pd.DataFrame(rows)
    total_slots = int(result_df["retrieved_count"].sum())
    duplicate_slots = int(result_df["duplicate_count"].sum())
    summary = {
        "method": name,
        "hit_at_3": float(result_df["hit_at_3"].mean()),
        "mrr": float(result_df["reciprocal_rank"].mean()),
        "duplicate_rate": duplicate_slots / total_slots if total_slots else 0.0,
        "avg_unique_cases": float(result_df["unique_case_count"].mean()),
    }
    return result_df, summary, details


def show_retrieval(details, evaluation_set):
    for item in evaluation_set:
        question = item["question"]
        print("=" * 100)
        print("질문:", question)
        print("기대 판례:", item["expected_case_no"])
        for rank, doc in enumerate(details[question], 1):
            print(
                f"{rank}. {doc.metadata.get('case_name')} "
                f"({doc.metadata.get('case_no')}, {doc.metadata.get('decision_date')})"
            )


## 3. Baseline 측정

Baseline은 판례 본문을 1,000자 단위로 분할하고 Vector Similarity Search Top-3를 사용한다. 개선 전 결과를 그대로 저장해 이후 단계의 기준으로 사용한다.


In [ ]:
baseline_eval, baseline_summary, baseline_details = evaluate_retrieval(
    "Baseline Vector",
    lambda question: retriever.invoke(question),
    evaluation_set,
)

display(baseline_eval)
display(pd.DataFrame([baseline_summary]))
show_retrieval(baseline_details, evaluation_set)


## 4. 개선 1: Context-Enriched Chunking

### 문제 진단

Baseline의 `page_content`에는 판례 본문만 들어 있다. 사건명·사건번호·선고일자는 Metadata에 있지만 임베딩 대상 텍스트에는 포함되지 않는다. 이 때문에 다음 문제가 생길 수 있다.

- 사건번호를 직접 묻는 질문이 본문 의미 검색에 제대로 반영되지 않는다.
- 짧은 청크만 보면 어느 판례에서 나온 내용인지 문맥이 약하다.
- 유사한 법률 표현을 사용하는 서로 다른 판례를 구분하기 어렵다.

### 적용 방법

기존 청크마다 사건명·사건번호·선고일자·법원명을 앞에 붙인다. 청크 크기와 임베딩 모델은 그대로 유지해, 문맥 정보 추가의 효과만 비교한다.


In [ ]:
from langchain_core.documents import Document


def enrich_chunk(doc):
    metadata = doc.metadata
    header = (
        f"[사건명] {metadata.get('case_name', '')}\n"
        f"[사건번호] {metadata.get('case_no', '')}\n"
        f"[선고일자] {metadata.get('decision_date', '')}\n"
        f"[법원명] {metadata.get('court', '')}\n"
        "[판례내용]\n"
    )
    return Document(
        page_content=header + doc.page_content,
        metadata={**metadata, "indexing": "context_enriched"},
    )


enriched_docs = [enrich_chunk(doc) for doc in docs]

print(f"Baseline Chunks: {len(docs):,}")
print(f"Enriched Chunks: {len(enriched_docs):,}")
print("\n[Enriched Chunk Sample]")
print(enriched_docs[0].page_content[:700])


In [ ]:
# Baseline Vector Store와 분리하여 동일 청크가 중복 적재되지 않게 한다.
enriched_collection_name = f"copyright_enriched_{uuid.uuid4().hex[:8]}"

enriched_vectorstore = Chroma.from_documents(
    documents=enriched_docs,
    embedding=embedding_model,
    collection_name=enriched_collection_name,
)

enriched_retriever = enriched_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

print("Context-Enriched Vector Store 생성 완료")
print(f"Collection: {enriched_collection_name}")
print(f"저장된 벡터 수: {enriched_vectorstore._collection.count():,}")


In [ ]:
enriched_eval, enriched_summary, enriched_details = evaluate_retrieval(
    "Enriched Vector",
    lambda question: enriched_retriever.invoke(question),
    evaluation_set,
)

display(enriched_eval)
display(pd.DataFrame([baseline_summary, enriched_summary]))
show_retrieval(enriched_details, evaluation_set)


### 개선 1 해석

다음 항목을 출력 결과에서 확인한다.

- 사건번호 질문의 기대 판례 순위가 올라갔는가?
- 기존에 찾지 못했던 판례가 Top-3에 포함됐는가?
- Metadata 텍스트 추가로 오히려 본문 의미 검색이 약해진 질문은 없는가?

결과가 모든 질문에서 좋아지지 않아도 정상이다. 어떤 질문이 개선되고 악화됐는지를 기록한다.


## 5. 개선 2: MMR와 사건번호 중복 제거

### 문제 진단

Vector Similarity Search는 질문과 가장 비슷한 청크부터 반환하므로, 하나의 판례에서 생성된 유사 청크가 Top-3를 모두 차지할 수 있다.

### 적용 방법

1. MMR로 관련성과 문서 다양성을 함께 고려한다.
2. `fetch_k=40`으로 후보를 넉넉히 가져온다.
3. 사건번호가 같은 청크에서는 가장 높은 순위 하나만 남긴다.
4. 최종 Top-3를 반환한다.


In [ ]:
enriched_mmr_retriever = enriched_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 12,
        "fetch_k": 40,
        "lambda_mult": 0.65,
    },
)


def deduplicate_by_case(docs, limit=3):
    selected = []
    seen = set()
    for doc in docs:
        case_no = case_no_of(doc)
        key = case_no or (doc.metadata.get("case_name", ""), doc.page_content[:100])
        if key in seen:
            continue
        seen.add(key)
        selected.append(doc)
        if len(selected) >= limit:
            break
    return selected


def mmr_dedup_retrieve(question):
    candidates = enriched_mmr_retriever.invoke(question)
    return deduplicate_by_case(candidates, limit=3)


mmr_eval, mmr_summary, mmr_details = evaluate_retrieval(
    "Enriched + MMR + Dedup",
    mmr_dedup_retrieve,
    evaluation_set,
)

display(mmr_eval)
display(pd.DataFrame([baseline_summary, enriched_summary, mmr_summary]))
show_retrieval(mmr_details, evaluation_set)


### 개선 2 해석

- Duplicate Rate가 감소했는가?
- 질문당 고유 판례 수가 증가했는가?
- 다양성을 높이는 과정에서 기대 판례 순위가 떨어진 질문은 없는가?
- `lambda_mult=0.65`가 현재 데이터에 적절한가는 별도 실험 대상이다.


## 6. 개선 3: BM25 + Vector Hybrid Search

### 문제 진단

법률 질문에는 두 종류의 검색이 모두 필요하다.

- Vector Search: 표현이 달라도 의미가 비슷한 판례 검색
- BM25: 사건번호, 공중송신권, 동일성유지권 같은 정확한 표현 검색

### 적용 방법

Enriched 청크에 BM25를 구성한다. BM25와 MMR Vector 후보의 순위를 Reciprocal Rank Fusion으로 결합한 후 사건번호 중복을 제거한다.


In [ ]:
import re
from langchain_community.retrievers import BM25Retriever


def legal_tokenize(text):
    normalized = re.sub(r"[^0-9A-Za-z가-힣]+", " ", str(text).lower()).strip()
    words = normalized.split()
    compact = "".join(words)
    # 한국어 띄어쓰기 차이를 완화하기 위해 글자 2-gram을 함께 사용한다.
    bigrams = [compact[i:i+2] for i in range(max(0, len(compact)-1))]
    return words + bigrams


bm25_retriever = BM25Retriever.from_documents(
    enriched_docs,
    preprocess_func=legal_tokenize,
)
bm25_retriever.k = 12


def chunk_key(doc):
    return case_no_of(doc), doc.page_content


def reciprocal_rank_fusion(result_lists, weights=None, rrf_k=60):
    weights = weights or [1.0] * len(result_lists)
    scores = {}
    document_map = {}

    for result, weight in zip(result_lists, weights):
        for rank, doc in enumerate(result, 1):
            key = chunk_key(doc)
            document_map[key] = doc
            scores[key] = scores.get(key, 0.0) + weight / (rrf_k + rank)

    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [document_map[key] for key in ranked_keys]


def hybrid_retrieve(question, limit=3):
    vector_candidates = enriched_mmr_retriever.invoke(question)
    keyword_candidates = bm25_retriever.invoke(question)
    fused = reciprocal_rank_fusion(
        [vector_candidates, keyword_candidates],
        weights=[0.6, 0.4],
    )
    return deduplicate_by_case(fused, limit=limit)


hybrid_eval, hybrid_summary, hybrid_details = evaluate_retrieval(
    "Enriched + MMR + Hybrid",
    hybrid_retrieve,
    evaluation_set,
)

display(hybrid_eval)
show_retrieval(hybrid_details, evaluation_set)


## 7. 전체 정량 비교

높을수록 좋은 지표는 `Hit@3`, `MRR`, `Average Unique Cases`이다. 낮을수록 좋은 지표는 `Duplicate Rate`이다. 하나의 지표만으로 최종 방법을 선택하지 않고 질문별 결과도 함께 확인한다.


In [ ]:
evaluation_summary = pd.DataFrame([
    baseline_summary,
    enriched_summary,
    mmr_summary,
    hybrid_summary,
])

display_summary = evaluation_summary.copy()
for column in ["hit_at_3", "mrr", "duplicate_rate"]:
    display_summary[column] = display_summary[column].map(lambda value: f"{value:.1%}")
display_summary["avg_unique_cases"] = display_summary["avg_unique_cases"].round(2)

display(display_summary)


In [ ]:
# 방법별 Retrieval 평가 결과 시각화
import matplotlib.pyplot as plt
import numpy as np

plot_df = evaluation_summary.copy().set_index("method")
methods = plot_df.index.tolist()
x = np.arange(len(methods))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 높을수록 좋은 지표
width = 0.36
axes[0].bar(x - width/2, plot_df["hit_at_3"], width, label="Hit@3", color="#2563EB")
axes[0].bar(x + width/2, plot_df["mrr"], width, label="MRR", color="#14B8A6")
axes[0].set_title("Retrieval accuracy by method")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.05)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=18, ha="right")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

# 낮을수록 좋은 중복률
bars = axes[1].bar(x, plot_df["duplicate_rate"], color="#F97316")
axes[1].set_title("Duplicate rate by method (lower is better)")
axes[1].set_ylabel("Duplicate rate")
axes[1].set_ylim(0, max(0.1, float(plot_df["duplicate_rate"].max()) * 1.25))
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=18, ha="right")
axes[1].grid(axis="y", alpha=0.25)

for bar, value in zip(bars, plot_df["duplicate_rate"]):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:.1%}",
        ha="center",
        va="bottom",
    )

fig.suptitle("Baseline and Advanced RAG retrieval comparison", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()


### 그래프 해석 방법

- 왼쪽 그래프에서 Hit@3과 MRR이 높아졌는지 확인한다.
- 오른쪽 그래프에서 중복률이 낮아졌는지 확인한다.
- 정확도와 중복률이 서로 다른 방향으로 움직였다면 하나의 지표만으로 최종 방법을 선택하지 않는다.
- 특정 방법이 전체 평균은 좋아도 개별 질문에서는 악화될 수 있으므로 바로 아래 질문별 비교표도 함께 확인한다.


In [ ]:
# 질문별로 어느 단계에서 개선 또는 악화됐는지 한 표에서 확인한다.
question_comparison = pd.concat([
    baseline_eval,
    enriched_eval,
    mmr_eval,
    hybrid_eval,
], ignore_index=True)

display(question_comparison[[
    "method",
    "question",
    "expected_case_no",
    "retrieved_case_nos",
    "hit_at_3",
    "expected_rank",
    "duplicate_count",
]])


## 8. 최종 답변 비교

가장 많은 기법을 적용한 방법이 항상 가장 좋은 것은 아니다. 위 평가 결과에서 Retrieval 성능과 중복률을 함께 확인한 뒤 최종 Retriever를 선택한다. 아래에서는 전체 기법을 적용한 Hybrid Retriever를 Baseline과 비교한다.


In [ ]:
from langchain_core.runnables import RunnableLambda

advanced_retriever_runnable = RunnableLambda(hybrid_retrieve)

advanced_rag_chain = (
    {
        "context": advanced_retriever_runnable | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("Advanced RAG Chain 구성 완료")


In [ ]:
comparison_rows = []

for item in evaluation_set:
    question = item["question"]
    baseline_answer = rag_chain.invoke(question)
    advanced_answer = advanced_rag_chain.invoke(question)
    baseline_cases = [case_no_of(doc) for doc in baseline_details[question]]
    advanced_cases = [case_no_of(doc) for doc in hybrid_details[question]]

    comparison_rows.append({
        "question": question,
        "expected_case_no": item["expected_case_no"],
        "baseline_cases": " | ".join(baseline_cases),
        "advanced_cases": " | ".join(advanced_cases),
        "baseline_answer": baseline_answer,
        "advanced_answer": advanced_answer,
    })

comparison_df = pd.DataFrame(comparison_rows)

for row in comparison_rows:
    print("=" * 100)
    print(f"[Question]\n{row['question']}")
    print(f"\n[Expected Case]\n{row['expected_case_no']}")
    print(f"\n[Baseline Cases]\n{row['baseline_cases']}")
    print(f"\n[Advanced Cases]\n{row['advanced_cases']}")
    print(f"\n[Baseline Answer]\n{row['baseline_answer']}")
    print(f"\n[Advanced Answer]\n{row['advanced_answer']}")

display(comparison_df[[
    "question", "expected_case_no", "baseline_cases", "advanced_cases"
]])


## 9. 제출용 결론 작성

아래 항목을 실제 실행 결과에 맞춰 작성한다. 결과를 실행하기 전에 수치를 미리 단정하지 않는다.

### Baseline 문제

- 기대 판례를 찾지 못한 질문:
- 동일 판례가 반복된 질문:
- 판례가 검색됐지만 기대 순위가 낮았던 질문:

### 단계별 결과

1. Context-Enriched Chunking으로 개선된 질문과 악화된 질문
2. MMR 적용 후 중복률과 고유 판례 수 변화
3. Hybrid Search 적용 후 정확한 법률 용어·사건번호 검색 변화

### 최종 판단

- 최종 선택 Retriever와 선택 이유
- 개선되지 않은 질문 및 추정 원인
- 다음 개선 후보: Query Rewriting, Reranking, Parent-Child Retrieval


In [ ]:
# 실행 결과에서 제출용 핵심 수치를 자동으로 정리한다.
best_hit = evaluation_summary.loc[evaluation_summary["hit_at_3"].idxmax()]
lowest_duplicate = evaluation_summary.loc[evaluation_summary["duplicate_rate"].idxmin()]
best_mrr = evaluation_summary.loc[evaluation_summary["mrr"].idxmax()]

print("[자동 요약]")
print(f"Hit@3 최고: {best_hit['method']} ({best_hit['hit_at_3']:.1%})")
print(f"MRR 최고: {best_mrr['method']} ({best_mrr['mrr']:.3f})")
print(
    f"중복률 최저: {lowest_duplicate['method']} "
    f"({lowest_duplicate['duplicate_rate']:.1%})"
)

print("\n[질문별 실패 사례]")
failures = question_comparison[question_comparison["hit_at_3"] == 0]
if failures.empty:
    print("모든 방법과 질문에서 기대 판례가 Top-3에 포함됐다.")
else:
    display(failures[["method", "question", "expected_case_no", "retrieved_case_nos"]])


# Final Summary

이번 실험에서는 Baseline의 검색 결과를 먼저 측정한 후, 문제 원인에 맞춰 Context-Enriched Chunking, MMR과 사건번호 중복 제거, BM25와 Vector Search의 Hybrid Retrieval을 순차적으로 적용했다.

각 기법은 모든 질문을 자동으로 개선하지 않는다. 따라서 같은 평가 질문으로 Hit@3, MRR, 중복률, 고유 판례 수를 비교하고 질문별 검색 결과와 최종 답변을 함께 확인했다. 법률 RAG에서는 자연스러운 답변보다 실제 근거 판례와 사건번호의 정확성을 우선해서 검증해야 한다.

최종 수치와 선택한 Retriever의 근거는 바로 위 실행 결과를 사용해 작성한다.


# 공식 Evaluation Set 비교

교수자가 배포한 `Copyright_RAG_Evaluation.xlsx`의 12문항 중 서로 다른 실패 유형을 확인할 수 있는 6문항을 선택한다.

| ID | 선택 이유 |
|---|---|
| E01 | 폰트와 프로그램 저작물 관련 법률 용어 검색 확인 |
| E03 | 권리 양도와 저작인격권을 구별하는 복합 질문 |
| E05 | 제품 사진의 창작성 판단과 과도한 일반화 확인 |
| E07 | 게임의 아이디어와 표현 구별 확인 |
| E10 | 동의 범위와 동일성유지권의 조건부 판단 확인 |
| E11 | 링크의 직접 침해와 방조를 구별하는 복합 판단 |

공식 파일에는 정답 판례 번호나 모범답안이 없다. 따라서 검색 중복도는 코드로 측정하고, 검색 관련성과 답변 품질은 동일한 평가 프롬프트로 비교한다. 최종 판단에서는 자동 점수뿐 아니라 실제 판례 제목·본문과 답변을 직접 확인한다.


## 10. 공식 평가 파일 업로드

`Copyright_RAG_Evaluation.xlsx`를 업로드한다. 앞서 판례 데이터 파일과 별도로 한 번 더 업로드 창이 열린다.


In [ ]:
uploaded_eval = files.upload()
EVAL_PATH = next(iter(uploaded_eval))

official_eval_df = pd.read_excel(EVAL_PATH)
required_columns = {"ID", "Category", "Question"}
missing_columns = required_columns - set(official_eval_df.columns)
if missing_columns:
    raise ValueError(f"평가 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

selected_ids = ["E01", "E03", "E05", "E07", "E10", "E11"]
selected_eval_df = (
    official_eval_df[official_eval_df["ID"].isin(selected_ids)]
    .set_index("ID")
    .loc[selected_ids]
    .reset_index()
)

print(f"공식 문항 수: {len(official_eval_df)}")
print(f"비교에 사용할 문항 수: {len(selected_eval_df)}")
display(selected_eval_df)


## 11. Baseline과 개선 모델 실행

공정한 비교를 위해 질문, Top-k, 답변 프롬프트와 LLM은 동일하게 유지하고 Retriever만 바꾼다.

- Baseline: Vector Similarity Search Top-3
- Advanced: Context-Enriched Chunking + MMR + 사건번호 중복 제거 + BM25/Vector Hybrid Search

API 비용을 줄이기 위해 대표 6문항만 사용한다.


In [ ]:
def case_list(docs):
    return [case_no_of(doc) for doc in docs]


def retrieval_stats(docs):
    case_numbers = [case_no for case_no in case_list(docs) if case_no]
    unique_count = len(set(case_numbers))
    duplicate_rate = 0.0 if not case_numbers else 1 - (unique_count / len(case_numbers))
    return unique_count, duplicate_rate


official_runs = []

for row in selected_eval_df.to_dict("records"):
    question = row["Question"]
    baseline_docs = retriever.invoke(question)
    advanced_docs = hybrid_retrieve(question)

    baseline_answer = rag_chain.invoke(question)
    advanced_answer = advanced_rag_chain.invoke(question)

    baseline_unique, baseline_dup = retrieval_stats(baseline_docs)
    advanced_unique, advanced_dup = retrieval_stats(advanced_docs)

    official_runs.append({
        "id": row["ID"],
        "category": row["Category"],
        "question": question,
        "baseline_docs": baseline_docs,
        "advanced_docs": advanced_docs,
        "baseline_cases": case_list(baseline_docs),
        "advanced_cases": case_list(advanced_docs),
        "baseline_unique_cases": baseline_unique,
        "advanced_unique_cases": advanced_unique,
        "baseline_duplicate_rate": baseline_dup,
        "advanced_duplicate_rate": advanced_dup,
        "baseline_answer": baseline_answer,
        "advanced_answer": advanced_answer,
    })

retrieval_comparison = pd.DataFrame([
    {
        "ID": run["id"],
        "Category": run["category"],
        "Baseline Cases": ", ".join(run["baseline_cases"]),
        "Advanced Cases": ", ".join(run["advanced_cases"]),
        "Baseline Unique": run["baseline_unique_cases"],
        "Advanced Unique": run["advanced_unique_cases"],
        "Baseline Duplicate Rate": run["baseline_duplicate_rate"],
        "Advanced Duplicate Rate": run["advanced_duplicate_rate"],
    }
    for run in official_runs
])

display(retrieval_comparison)


## 12. Retrieval 및 Answer Quality 평가

정답 레이블이 없는 평가셋이므로 다음 루브릭을 같은 평가 모델에 적용한다.

- Retrieval Relevance(1~5): 검색 판례가 질문의 쟁점과 직접 관련되는 정도
- Answer Relevance(1~5): 질문에 직접 답했는가
- Groundedness(1~5): 검색된 판례에 근거하며 근거 없는 단정이 없는가
- Completeness(1~5): 핵심 조건과 예외를 빠뜨리지 않았는가

LLM 평가는 절대적 정답이 아니라 비교를 돕는 보조 지표다. `reason`과 원문 결과를 함께 검토한다.


In [ ]:
from typing import List
from pydantic import BaseModel, Field


class ModelScore(BaseModel):
    retrieval_relevance: int = Field(ge=1, le=5)
    answer_relevance: int = Field(ge=1, le=5)
    groundedness: int = Field(ge=1, le=5)
    completeness: int = Field(ge=1, le=5)
    reason: str


class PairEvaluation(BaseModel):
    baseline: ModelScore
    advanced: ModelScore
    better_model: str = Field(description="baseline, advanced, tie 중 하나")
    comparison_reason: str


judge_prompt = ChatPromptTemplate.from_template("""
당신은 RAG 시스템 평가자입니다. 동일한 질문에 대한 Baseline과 Advanced 결과를 아래 기준으로 각각 1~5점으로 평가하세요.

[평가 기준]
1. retrieval_relevance: 검색 판례가 질문의 법률 쟁점과 직접 관련되는 정도
2. answer_relevance: 질문에 직접 답하는 정도
3. groundedness: 검색 판례에 근거하고 근거 없는 단정이 없는 정도
4. completeness: 핵심 조건, 예외, 구별 기준을 포함한 정도

두 모델에 같은 기준을 적용하세요. 제공되지 않은 정답 판례나 외부 법률 지식을 사실처럼 추가하지 마세요.

[Question]
{question}

[Baseline Retrieved Cases]
{baseline_context}

[Baseline Answer]
{baseline_answer}

[Advanced Retrieved Cases]
{advanced_context}

[Advanced Answer]
{advanced_answer}
""")

judge = llm.with_structured_output(PairEvaluation)
judge_chain = judge_prompt | judge

quality_rows = []
for run in official_runs:
    judged = judge_chain.invoke({
        "question": run["question"],
        "baseline_context": format_docs(run["baseline_docs"]),
        "baseline_answer": run["baseline_answer"],
        "advanced_context": format_docs(run["advanced_docs"]),
        "advanced_answer": run["advanced_answer"],
    })

    quality_rows.extend([
        {
            "ID": run["id"], "Method": "Baseline",
            "Retrieval Relevance": judged.baseline.retrieval_relevance,
            "Answer Relevance": judged.baseline.answer_relevance,
            "Groundedness": judged.baseline.groundedness,
            "Completeness": judged.baseline.completeness,
            "Reason": judged.baseline.reason,
            "Better Model": judged.better_model,
            "Comparison Reason": judged.comparison_reason,
        },
        {
            "ID": run["id"], "Method": "Advanced",
            "Retrieval Relevance": judged.advanced.retrieval_relevance,
            "Answer Relevance": judged.advanced.answer_relevance,
            "Groundedness": judged.advanced.groundedness,
            "Completeness": judged.advanced.completeness,
            "Reason": judged.advanced.reason,
            "Better Model": judged.better_model,
            "Comparison Reason": judged.comparison_reason,
        },
    ])

quality_comparison = pd.DataFrame(quality_rows)
display(quality_comparison)


## 13. 공식 평가 결과 요약과 시각화

평균 점수만 보면 특정 질문의 악화를 놓칠 수 있다. 전체 평균과 문항별 변화량을 함께 확인한다.


In [ ]:
quality_metrics = [
    "Retrieval Relevance", "Answer Relevance", "Groundedness", "Completeness"
]

quality_summary = (
    quality_comparison.groupby("Method")[quality_metrics]
    .mean()
    .round(2)
)

duplicate_summary = pd.DataFrame({
    "Method": ["Baseline", "Advanced"],
    "Average Unique Cases": [
        retrieval_comparison["Baseline Unique"].mean(),
        retrieval_comparison["Advanced Unique"].mean(),
    ],
    "Average Duplicate Rate": [
        retrieval_comparison["Baseline Duplicate Rate"].mean(),
        retrieval_comparison["Advanced Duplicate Rate"].mean(),
    ],
}).set_index("Method").round(3)

display(quality_summary)
display(duplicate_summary)

ax = quality_summary.T.plot(kind="bar", figsize=(10, 5), ylim=(0, 5), rot=0)
ax.set_title("Baseline vs Advanced: Official Evaluation Set")
ax.set_ylabel("Average Score (1-5)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
score_wide = quality_comparison.pivot(index="ID", columns="Method", values=quality_metrics)
delta_rows = []

for eval_id in selected_ids:
    deltas = {
        metric: score_wide.loc[eval_id, (metric, "Advanced")]
        - score_wide.loc[eval_id, (metric, "Baseline")]
        for metric in quality_metrics
    }
    delta_rows.append({
        "ID": eval_id,
        **{f"Delta {metric}": value for metric, value in deltas.items()},
        "Average Delta": sum(deltas.values()) / len(deltas),
    })

delta_df = pd.DataFrame(delta_rows).sort_values("Average Delta", ascending=False)
improved_cases = delta_df[delta_df["Average Delta"] > 0]
worsened_cases = delta_df[delta_df["Average Delta"] < 0]
unchanged_cases = delta_df[delta_df["Average Delta"] == 0]

print("[개선된 사례]")
display(improved_cases)
print("[악화된 사례]")
display(worsened_cases)
print("[변화 없는 사례]")
display(unchanged_cases)

display_columns = ["ID", "Method", "Reason", "Better Model", "Comparison Reason"]
display(quality_comparison[display_columns])


## 14. 제출용 해석 작성

실행 결과를 확인한 뒤 아래 문장을 실제 수치와 사례로 완성한다.

### 기법 선택 이유

1. **Context-Enriched Chunking**: 판례 본문만으로 부족한 사건명·사건번호·선고일자 문맥을 임베딩 텍스트에 포함하기 위해 선택했다.
2. **MMR + 사건번호 중복 제거**: 같은 판례의 유사 청크가 Top-3를 반복 점유해 다른 관련 판례를 배제하는 문제를 줄이기 위해 선택했다.
3. **BM25 + Vector Hybrid Search**: 의미가 유사한 표현은 Vector Search로, 사건번호와 법률 용어의 정확 일치는 BM25로 보완하기 위해 선택했다.

### 결과 기록

- Baseline 평균 Retrieval Relevance: `[실행값]`
- Advanced 평균 Retrieval Relevance: `[실행값]`
- Baseline 평균 Answer Quality: `[실행값]`
- Advanced 평균 Answer Quality: `[실행값]`
- 검색 중복률 변화: `[실행값] → [실행값]`
- 가장 개선된 문항과 이유: `[ID와 관찰 내용]`
- 개선되지 않았거나 악화된 문항과 이유: `[ID와 관찰 내용]`

### 해석 시 주의점

- 평균 점수가 상승해도 모든 질문이 개선됐다고 표현하지 않는다.
- Advanced가 악화된 문항은 Hybrid 가중치, MMR 다양성, 검색된 판례 내용을 직접 확인한다.
- LLM Judge의 점수는 보조 지표이므로 검색 판례와 답변 원문을 함께 제시한다.
- 법률 답변의 정확성을 보장하는 시스템이 아니라 제공 판례 범위에서 답을 생성하는 실험임을 명시한다.
